In [2]:
#imports
import pandas as pd
import numpy as np
import os 
import geopandas as gpd

In [7]:
input_dir= "../../Cleaned_Data"


In [13]:
climate_df= pd.read_csv(os.path.join(input_dir,"climate_2024_imputed.csv"))
soil_df= pd.read_csv(os.path.join(input_dir,"soil_properties_imputed.csv"))

In [14]:
soil_df.columns

Index(['y', 'x', 'HWSD2_SMU_ID', 'COARSE', 'SAND', 'SILT', 'CLAY', 'BULK',
       'REF_BULK', 'ORG_CARBON', 'PH_WATER', 'TOTAL_N', 'CEC_SOIL', 'CEC_CLAY',
       'CEC_EFF', 'TEB', 'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM',
       'ELEC_COND'],
      dtype='object')

In [15]:
climate_df.columns

Index(['cell_id', 'tmin_1_2024', 'tmax_1_2024', 'prec_1_2024', 'tmin_2_2024',
       'tmax_2_2024', 'prec_2_2024', 'tmin_3_2024', 'tmax_3_2024',
       'prec_3_2024', 'tmin_4_2024', 'tmax_4_2024', 'prec_4_2024'],
      dtype='object')

In [16]:
# generate the polygon with geopandas
soil_gdf = gpd.GeoDataFrame(
    soil_df, geometry=gpd.points_from_xy(soil_df.x, soil_df.y), crs="EPSG:4326"
)

In [17]:
# Generate polygon for climate data
# If climate_df doesn't have spatial coordinates, we need to load the grid
grid_path = "../../ExtractedDatasets/ClimateFeatures/grid_0p1.gpkg"
grid = gpd.read_file(grid_path, layer="grid")

# Add centroids to grid
grid['centroid'] = grid.geometry.centroid
grid['lon'] = grid['centroid'].x
grid['lat'] = grid['centroid'].y

# Merge climate data with grid to get coordinates
climate_df = climate_df.merge(grid[['cell_id', 'lon', 'lat', 'geometry']], on='cell_id', how='left')
print("Merged climate data with grid. New shape:", climate_df.shape)

# Create GeoDataFrame from climate data
climate_gdf = gpd.GeoDataFrame(
    climate_df, 
    geometry=gpd.points_from_xy(climate_df.lon, climate_df.lat), 
    crs="EPSG:4326"
)

print("\nClimate GeoDataFrame created:")
print(f"Shape: {climate_gdf.shape}")
print(f"CRS: {climate_gdf.crs}")


Merged climate data with grid. New shape: (23322, 16)

Climate GeoDataFrame created:
Shape: (23322, 16)
CRS: EPSG:4326


C:\Users\E15\AppData\Local\Temp\ipykernel_5336\1942890913.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid['centroid'] = grid.geometry.centroid


In [ ]:
climate_df.columns

In [22]:
## check if x y in soil and lon lat in climate are the same
soil_df[['x','y']] == climate_df[['lon','lat']]


ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

In [ ]:
# Reduce climate_df to only those cell_ids present in merged
df_climate_reduced = climate_df[climate_df['cell_id'].isin(merged['cell_id'])]
print(f"Reduced climate data shape: {df_climate_reduced.shape}")
